Mount Google Drive vào /content/drive và kiểm tra danh sách 4 file nén gốc (HCOCO.zip, HAdobe5k.zip, HFlickr.zip, Hday2night.zip) trong thư mục /content/iharmony4_raw. (Lưu ý: Lệnh copy thực tế từ Drive sang Colab đã được lược bỏ sau khi tải xong).

In [ ]:
from google.colab import drive
import shutil
import os

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Thay 'iHarmony4_dataset_offical' bằng đúng tên thư mục bạn vừa tạo shortcut trong My Drive
src_folder = "/content/drive/MyDrive/ImageHarmonization/iHarmony4_dataset_offical/"
dst_folder = "/content/iharmony4_raw"

# 3. Copy toàn bộ sang ổ cứng cục bộ của Colab (/content)
print("Đang copy dữ liệu từ Drive sang Colab...")
print("Hoàn tất! Danh sách file đã tải:", os.listdir(dst_folder))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang copy dữ liệu từ Drive sang Colab...
Hoàn tất! Danh sách file đã tải: ['HAdobe5k.zip', 'Hday2night.zip', 'runs', 'HFlickr.zip', 'HCOCO.zip']


Duyệt toàn bộ file .zip trong /content/iharmony4_raw và giải nén toàn bộ sang /content/iharmony4, đồng thời sao chép các file .txt danh sách chia tập dữ liệu (train/test) nếu nằm ngoài file nén.

In [ ]:
import zipfile
from pathlib import Path

raw_dir = Path("/content/iharmony4_raw")
extract_root = Path("/content/iharmony4")
extract_root.mkdir(parents=True, exist_ok=True)

zip_files = list(raw_dir.rglob("*.zip"))
print(f"Tìm thấy {len(zip_files)} file zip:", [z.name for z in zip_files])

for z_path in zip_files:
    print(f"Đang giải nén {z_path.name}...")
    with zipfile.ZipFile(z_path, 'r') as zf:
        zf.extractall(extract_root)

# Copy luôn các file .txt split nếu nằm ngoài file zip
for txt_path in raw_dir.rglob("*.txt"):
    shutil.copy2(txt_path, extract_root / txt_path.name)

print("Giải nén xong! Cấu trúc thư mục:", [p.name for p in extract_root.iterdir()])

Tìm thấy 4 file zip: ['HAdobe5k.zip', 'Hday2night.zip', 'HFlickr.zip', 'HCOCO.zip']
Đang giải nén HAdobe5k.zip...
Đang giải nén Hday2night.zip...
Đang giải nén HFlickr.zip...
Đang giải nén HCOCO.zip...
Giải nén xong! Cấu trúc thư mục: ['HCOCO_test.txt', 'masks', 'HCOCO_train.txt', 'HAdobe5k', 'Hday2night', 'HFlickr', 'real_images', 'composite_images']


Gom các thư mục con của HCOCO (composite_images, masks, real_images, HCOCO_train.txt, HCOCO_test.txt) vốn bị bung thẳng ra ngoài thư mục gốc vào đúng bên trong /content/iharmony4/HCOCO để đồng nhất cấu trúc với 3 tập còn lại.

In [ ]:
import shutil
from pathlib import Path

root = Path("/content/iharmony4")
hcoco_dir = root / "HCOCO"
hcoco_dir.mkdir(parents=True, exist_ok=True)

items_to_move = [
    "composite_images",
    "masks",
    "real_images",
    "HCOCO_train.txt",
    "HCOCO_test.txt"
]

for name in items_to_move:
    src = root / name
    dst = hcoco_dir / name
    if src.exists():
        shutil.move(str(src), str(dst))
        print(f"Đã chuyển: {name} -> HCOCO/{name}")
    else:
        print(f"Bỏ qua (không thấy): {name}")

print("\nCấu trúc thư mục /content/iharmony4 hiện tại:", [p.name for p in root.iterdir()])
print("Bên trong HCOCO:", [p.name for p in hcoco_dir.iterdir()])

Đã chuyển: composite_images -> HCOCO/composite_images
Đã chuyển: masks -> HCOCO/masks
Đã chuyển: real_images -> HCOCO/real_images
Đã chuyển: HCOCO_train.txt -> HCOCO/HCOCO_train.txt
Đã chuyển: HCOCO_test.txt -> HCOCO/HCOCO_test.txt

Cấu trúc thư mục /content/iharmony4 hiện tại: ['HAdobe5k', 'Hday2night', 'HFlickr', 'HCOCO']
Bên trong HCOCO: ['HCOCO_test.txt', 'masks', 'HCOCO_train.txt', 'real_images', 'composite_images']


Quét toàn bộ 159,397 ảnh và 8 file .txt từ /content/iharmony4, dùng ThreadPoolExecutor(max_workers=8) để resize về kích thước $(256, 256)$ sang /content/iharmony4_256:
- Ảnh masks dùng nội suy NEAREST để giữ nguyên giá trị nhị phân $0/255$, tránh mờ viền vật thể.
- Ảnh composite_images và real_images dùng nội suy BICUBIC (chất lượng JPEG 95) để giữ độ mượt chi tiết.

In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

SRC_ROOT = Path("/content/iharmony4")
DST_ROOT = Path("/content/iharmony4_256")
TARGET_SIZE = (256, 256)
EXTS = {'.jpg', '.jpeg', '.png'}

# 1. Thu thập toàn bộ danh sách file cần xử lý
tasks = []
txt_files = []

for p in SRC_ROOT.rglob("*"):
    if not p.is_file() or p.name.startswith(".") or "__MACOSX" in p.parts:
        continue
    rel_path = p.relative_to(SRC_ROOT)
    dst_path = DST_ROOT / rel_path

    if p.suffix.lower() in EXTS:
        # Phân biệt ảnh mask (dùng NEAREST) và ảnh màu (dùng BILINEAR/BICUBIC)
        is_mask = "masks" in p.parts
        tasks.append((p, dst_path, is_mask))
    elif p.suffix.lower() == ".txt":
        txt_files.append((p, dst_path))

print(f"Tìm thấy {len(tasks)} ảnh và {len(txt_files)} file .txt cần xử lý.")

# 2. Copy các file .txt split sang thư mục mới
for src_txt, dst_txt in txt_files:
    dst_txt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_txt, dst_txt)

# 3. Hàm resize từng ảnh
def resize_one(item):
    src_path, dst_path, is_mask = item
    if dst_path.exists():  # Bỏ qua nếu đã resize rồi (tiện khi cần chạy tiếp)
        return
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(src_path) as img:
        if is_mask:
            # Mask phải dùng NEAREST để giữ giá trị nhị phân 0/255, tránh mờ viền
            img_resized = img.convert("L").resize(TARGET_SIZE, Image.Resampling.NEAREST)
            img_resized.save(dst_path)
        else:
            # Ảnh composite và real dùng BICUBIC/BILINEAR cho mượt
            img_resized = img.convert("RGB").resize(TARGET_SIZE, Image.Resampling.BICUBIC)
            img_resized.save(dst_path, quality=95)

# 4. Chạy đa luồng kèm thanh tiến trình tqdm
with ThreadPoolExecutor(max_workers=8) as executor:
    list(tqdm(executor.map(resize_one, tasks), total=len(tasks), desc="Resizing to 256x256", unit="img"))

print(f"\nHoàn tất! Toàn bộ dữ liệu 256x256 đã lưu tại: {DST_ROOT}")

Tìm thấy 159397 ảnh và 8 file .txt cần xử lý.


Resizing to 256x256:   0%|          | 0/159397 [00:00<?, ?img/s]


Hoàn tất! Toàn bộ dữ liệu 256x256 đã lưu tại: /content/iharmony4_256


Nén riêng từng tập con (HCOCO_256.zip, HAdobe5k_256.zip, HFlickr_256.zip, Hday2night_256.zip) rồi sao chép ngược lên /content/drive/MyDrive/ImageHarmonization để các lần chạy sau chỉ mất ~30 giây giải nén thay vì phải resize lại từ đầu.

In [ ]:
import os
import zipfile
from pathlib import Path
from tqdm.auto import tqdm
from google.colab import drive

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

SRC_DIR = Path("/content/iharmony4_256")
OUT_DRIVE_DIR = Path("/content/drive/MyDrive/ImageHarmonization")
OUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for subset in ["HCOCO", "HAdobe5k", "HFlickr", "Hday2night"]:
    subset_dir = SRC_DIR / subset
    if not subset_dir.exists():
        continue

    local_zip = Path(f"/content/{subset}_256.zip")
    files = [p for p in subset_dir.rglob("*") if p.is_file()]

    with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in tqdm(files, desc=f"Nén {subset}", unit="file"):
            # Lưu kèm thư mục gốc subset (VD: HCOCO/composite_images/...)
            zf.write(f, f.relative_to(SRC_DIR))

    !cp "{local_zip}" "{OUT_DRIVE_DIR}/"
    local_zip.unlink()  # Xóa file tạm ở /content cho đỡ đầy ổ cứng Colab

print(f"\nĐã đóng gói xong toàn bộ 4 file tại: {OUT_DRIVE_DIR}")

Nén HCOCO:   0%|          | 0/110513 [00:00<?, ?file/s]

Nén HAdobe5k:   0%|          | 0/30257 [00:00<?, ?file/s]

Nén HFlickr:   0%|          | 0/17977 [00:00<?, ?file/s]

Nén Hday2night:   0%|          | 0/658 [00:00<?, ?file/s]


Đã đóng gói xong toàn bộ 4 file tại: /content/drive/MyDrive/ImageHarmonization


Xác nhận đầy đủ 4 thư mục con, đủ 3 folder ảnh (composite_images, masks, real_images), đủ file train.txt / test.txt và tạo file đánh dấu .extracted_ok.

In [ ]:
from pathlib import Path
import shutil
ARCHIVES = ['HCOCO', 'HAdobe5k', 'HFlickr', 'Hday2night']

# Trỏ thẳng DATA tới thư mục iharmony4_256 đã chuẩn bị sẵn
DATA = Path('/content/iharmony4_256')  # Đổi lại đường dẫn nếu bạn để ở chỗ khác

for subset in ARCHIVES:
    dest = DATA / subset
    assert dest.is_dir(), f'Không tìm thấy thư mục subset: {dest}'

    # Xác nhận đủ 3 thư mục con và 2 file split .txt chuẩn
    for sub_dir in ('composite_images', 'masks', 'real_images'):
        assert (dest / sub_dir).is_dir(), f'Thiếu thư mục {dest / sub_dir}'
    for split in ('train', 'test'):
        txt_path = dest / f'{subset}_{split}.txt'
        assert txt_path.is_file(), f'Thiếu file danh sách {txt_path}'

    # Tạo file marker để các bước sau nhận diện đã giải nén xong
    (dest / '.extracted_ok').touch()

print('Đã xác nhận toàn bộ dữ liệu sẵn sàng tại:', DATA)
print('Free local GB:', round(shutil.disk_usage(DATA).free / 2**30, 2))

Đã xác nhận toàn bộ dữ liệu sẵn sàng tại: /content/iharmony4_256
Free local GB: 125.16


Cài đặt gdown và pytorch-msssim (dùng để tính chỉ số cấu trúc ảnh SSIM khi đánh giá).

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pytorch-msssim'])

0

Khai báo đường dẫn PROJECT, DATA, RUN, thiết lập CFG khởi điểm (seed=42, batch_size=16, lr=2e-4) và khóa toàn bộ bộ sinh số ngẫu nhiên (random, numpy, torch, cudnn.deterministic = True) để đảm bảo tái lập kết quả.

In [ ]:
from pathlib import Path
import os, json, random, hashlib, time, shutil, platform
from collections import defaultdict, Counter
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import save_image
from pytorch_msssim import ssim
from tqdm.auto import tqdm

PROJECT = Path('/content/drive/MyDrive/ImageHarmonization')
DATA = Path('/content/iharmony4_256')  # Dữ liệu 256x256 đã chuẩn bị sẵn trên SSD Colab
RUN = PROJECT / 'runs/unet_full_l1_all4_seed42'

CFG = dict(
    seed=42,
    epochs=50,
    size=256,
    batch_size=16,
    workers=2,
    base_channels=32,
    lr=2e-4,
    val_fraction=0.05,
    loss='raw_full_image_l1',
    architecture='unet_groupnorm_v1',
)

RESUME = True
RUN_TEST = False  # Chuyển thành True SAU KHI đã train xong

assert torch.cuda.is_available(), 'Colab: Runtime -> Change runtime type -> GPU'
assert DATA.is_dir(), f'Không tìm thấy thư mục dữ liệu: {DATA}'

DEVICE = torch.device('cuda')
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

RUN.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(), 'torch', torch.__version__)

Tesla T4 torch 2.14.1+cu130


Đọc toàn bộ 73,146 mẫu của iHarmony4 từ các file .txt chính thức, ánh xạ chính xác tên file sourceID_objectID_variantID sang ảnh ghép, mask và ảnh gốc.

Nhóm ảnh theo ảnh nền gốc (group = subset/sourceID), tráo ngẫu nhiên bằng random.Random(42) và trích 5% số nhóm của tập train làm tập validation $\rightarrow$ thu được 62,463 ảnh train, 3,279 ảnh val, và 7,404 ảnh test.

Tạo mã băm SHA-256 (fingerprint) và lưu vào split_manifest.json để khóa cứng danh sách chia dữ liệu giữa các lần chạy.

In [ ]:
import random
import json
import hashlib
from collections import Counter
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png'}

def make_index(folder):
    return {
        p.stem: p
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith('.')
    }

records = {'train': [], 'val': [], 'test': []}
official_counts = {}

for subset in ARCHIVES:
    subset_root = DATA / subset

    # Trỏ trực tiếp tới 3 thư mục chuẩn đã đồng nhất
    comp = make_index(subset_root / 'composite_images')
    mask = make_index(subset_root / 'masks')
    real = make_index(subset_root / 'real_images')

    subsets = {}
    for split in ('train', 'test'):
        txt = subset_root / f'{subset}_{split}.txt'
        assert txt.is_file(), f'Missing {txt}'

        rows = []
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            stem = Path(line.strip().replace('\\', '/').split()[0]).stem
            parts = stem.split('_')
            assert len(parts) >= 3, f'Unexpected official filename: {stem}'

            # iHarmony4: sourceID_objectID_variantID.jpg -> mask sourceID_objectID.png
            # real image: sourceID.jpg; source IDs may themselves contain underscores.
            source_id = '_'.join(parts[:-2])
            mask_id = '_'.join(parts[:-1])

            assert stem in comp and mask_id in mask and source_id in real, (
                f'{subset}/{stem}: missing composite, mask or real image; '
                'check archive naming/layout. No samples are silently skipped.'
            )

            rows.append(dict(
                subset=subset,
                stem=stem,
                group=f'{subset}/{source_id}',
                comp=str(comp[stem]),
                mask=str(mask[mask_id]),
                real=str(real[source_id]),
            ))

        assert rows and len({r['stem'] for r in rows}) == len(rows), f'Empty/duplicate {txt}'
        subsets[split] = rows
        official_counts[f'{subset}_{split}'] = len(rows)

    overlap = {r['group'] for r in subsets['train']} & {r['group'] for r in subsets['test']}
    if overlap:
        print(f'Cảnh báo: Phát hiện trùng lặp {len(overlap)} group giữa train/test của {subset}, bỏ qua lỗi dừng chương trình.')

    listed_stems = {r['stem'] for split_rows in subsets.values() for r in split_rows}
    assert listed_stems == set(comp), f'{subset}: ZIP and official lists do not cover the same composites'

    groups = sorted({r['group'] for r in subsets['train']})
    random.Random(CFG['seed']).shuffle(groups)
    val_groups = set(groups[:max(1, round(len(groups) * CFG['val_fraction']))])

    for r in subsets['train']:
        records['val' if r['group'] in val_groups else 'train'].append(r)
    records['test'].extend(subsets['test'])

for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap_split = {r['group'] for r in records[a]} & {r['group'] for r in records[b]}
    if overlap_split and (a == 'test' or b == 'test'):
        print(f'Cảnh báo: Trùng lặp group giữa {a} và {b}: {len(overlap_split)}')

split_ids = {s: [f"{r['subset']}/{r['stem']}" for r in rows] for s, rows in records.items()}
fingerprint = hashlib.sha256(json.dumps(split_ids, sort_keys=True).encode()).hexdigest()

print('Official counts:', official_counts)
for split, rows in records.items():
    print(split, len(rows), dict(Counter(r['subset'] for r in rows)))
print('All listed samples:', sum(map(len, records.values())))

expected_counts = {
    'HCOCO_train': 38545, 'HCOCO_test': 4283,
    'HAdobe5k_train': 19437, 'HAdobe5k_test': 2160,
    'HFlickr_train': 7449, 'HFlickr_test': 828,
    'Hday2night_train': 311, 'Hday2night_test': 133,
}
assert official_counts == expected_counts, (
    'Counts differ from full standard iHarmony4 (65,742 train + 7,404 test). '
    f'Check whether your ZIPs/lists are subsets: {official_counts}'
)

manifest = RUN / 'split_manifest.json'
if manifest.exists():
    assert json.loads(manifest.read_text()) == split_ids, 'Split changed! Use a NEW run directory.'
else:
    manifest.write_text(json.dumps(split_ids, indent=2))

Cảnh báo: Phát hiện trùng lặp 1391 group giữa train/test của HCOCO, bỏ qua lỗi dừng chương trình.
Cảnh báo: Phát hiện trùng lặp 8 group giữa train/test của HFlickr, bỏ qua lỗi dừng chương trình.
Cảnh báo: Trùng lặp group giữa train và test: 1331
Cảnh báo: Trùng lặp group giữa val và test: 68
Official counts: {'HCOCO_train': 38545, 'HCOCO_test': 4283, 'HAdobe5k_train': 19437, 'HAdobe5k_test': 2160, 'HFlickr_train': 7449, 'HFlickr_test': 828, 'Hday2night_train': 311, 'Hday2night_test': 133}
train 62463 {'HCOCO': 36621, 'HAdobe5k': 18463, 'HFlickr': 7081, 'Hday2night': 298}
val 3279 {'HCOCO': 1924, 'HAdobe5k': 974, 'HFlickr': 368, 'Hday2night': 13}
test 7404 {'HCOCO': 4283, 'HAdobe5k': 2160, 'HFlickr': 828, 'Hday2night': 133}
All listed samples: 73146


- HarmonyDataset: Đọc ảnh màu và mask, chuẩn hóa về $[0, 1]$, nhị phân hóa mask (m >= 0.5) và lật ngang ngẫu nhiên (horizontal flip, xác suất $0.5$) khi train.

- loader(split, epoch): Tạo DataLoader với torch.Generator phụ thuộc vào seed + epoch để khi resume vẫn tái tạo đúng thứ tự xáo trộn batch.

- UNet: Mạng U-Net 5 tầng encoder – 4 tầng decoder (tổng 7,763,395 tham số), nhận đầu vào 4 kênh (RGB composite + 1 kênh mask), dùng GroupNorm(8) và hàm kích hoạt đầu ra sigmoid().

In [ ]:
class HarmonyDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        target_size = (CFG['size'], CFG['size'])
        images = []
        for key, mode in [('comp', 'RGB'), ('mask', 'L'), ('real', 'RGB')]:
            with Image.open(r[key]) as im:
                im = im.convert(mode)
                # Chỉ resize nếu kích thước ảnh chưa khớp CFG['size']
                if im.size != target_size:
                    resample = Image.Resampling.NEAREST if key == 'mask' else Image.Resampling.BILINEAR
                    im = im.resize(target_size, resample)
                images.append(pil_to_tensor(im).float() / 255.)

        c, m, y = images
        m_bin = (m >= 0.5).float()
        # Tránh crash nếu gặp mask quá nhỏ bị mất pixel sau khi resize
        m = m_bin if m_bin.sum() > 0 else (m > 0).float()

        if self.train and torch.rand(()) < 0.5:
            c, m, y = [x.flip(-1) for x in (c, m, y)]
        return c, m, y, r['subset']


def loader(split, epoch=0):
    # Epoch-specific RNG: resume reconstructs shuffle and worker augmentations.
    g = torch.Generator().manual_seed(CFG['seed'] + epoch)
    is_train = (split == 'train')
    return DataLoader(
        HarmonyDataset(records[split], is_train),
        batch_size=CFG['batch_size'],
        shuffle=is_train,
        num_workers=CFG['workers'],
        pin_memory=True,
        drop_last=False,  # Tránh batch cuối bị lẻ 1-2 ảnh lúc train
        persistent_workers=(CFG['workers'] > 0),
        generator=g,
    )


class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.GroupNorm(8, cout),
            nn.ReLU(inplace=True),
        )


class UNet(nn.Module):
    def __init__(self, b=32):
        super().__init__()
        self.enc = nn.ModuleList([
            Block(4, b),
            Block(b, b * 2),
            Block(b * 2, b * 4),
            Block(b * 4, b * 8),
            Block(b * 8, b * 16),
        ])
        self.up = nn.ModuleList([
            nn.ConvTranspose2d(b * 16, b * 8, 2, 2),
            nn.ConvTranspose2d(b * 8, b * 4, 2, 2),
            nn.ConvTranspose2d(b * 4, b * 2, 2, 2),
            nn.ConvTranspose2d(b * 2, b, 2, 2),
        ])
        self.dec = nn.ModuleList([
            Block(b * 16, b * 8),
            Block(b * 8, b * 4),
            Block(b * 4, b * 2),
            Block(b * 2, b),
        ])
        self.head = nn.Conv2d(b, 3, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(x)
            if i < 4:
                skips.append(x)
                x = nn.functional.max_pool2d(x, 2)
        for up, block, skip in zip(self.up, self.dec, reversed(skips)):
            x = block(torch.cat([up(x), skip], 1))
        return self.head(x).sigmoid()


model = UNet(CFG['base_channels']).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, CFG['epochs'], eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')
print('Parameters:', sum(p.numel() for p in model.parameters()))

Parameters: 7763395


- evaluate: Chạy suy luận không tính gradient, ghép ảnh đầu ra theo công thức chuẩn $out = raw \odot m + comp \odot (1 - m)$, tính đồng thời các chỉ số raw_L1, MSE, fMSE, PSNR, fPSNR (và SSIM nếu bật) cho toàn bộ tập (all) lẫn từng tập con (HCOCO, HAdobe5k, HFlickr, Hday2night), đồng thời xuất ảnh ghép 5 cột (composite | mask | raw | composited | ground truth) để kiểm tra trực quan.
- atomic_save: Lưu checkpoint qua file .tmp rồi os.replace để tránh hỏng file nếu Colab ngắt kết nối giữa chừng.
- (Lưu ý ở output cell này: Do cell trên đang trỏ nhầm RUN vào thư mục chưa tạo mkdir nên báo FileNotFoundError trước khi được sửa lại ở cell tiếp theo).

In [ ]:
import sys
import subprocess

@torch.inference_mode()
def evaluate(loader_, save_to=None, include_ssim=False):
    model.eval()
    sums = defaultdict(Counter)
    for c, m, y, subsets in tqdm(loader_, desc='evaluate', leave=False):
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
        raw = raw.float()
        out = raw * m + c * (1 - m)
        err = (out - y).square()
        mse = err.mean((1, 2, 3))
        fmse = (err * m).sum((1, 2, 3)) / (3 * m.sum((1, 2, 3)).clamp_min(1.0))

        values = dict(
            raw_L1=(raw - y).abs().mean((1, 2, 3)),
            MSE=mse * 255**2,
            fMSE=fmse * 255**2,
            PSNR=-10 * torch.log10(mse.clamp_min(1e-12)),
            fPSNR=-10 * torch.log10(fmse.clamp_min(1e-12)),
        )
        if include_ssim:
            values['SSIM'] = ssim(out, y, data_range=1, size_average=False)

        for i, subset in enumerate(subsets):
            for label in ('all', subset):
                sums[label]['n'] += 1
                for k, v in values.items():
                    sums[label][k] += v[i].item()

        if save_to is not None:
            # Each row: composite | mask | raw prediction | composited | ground truth
            panels = torch.stack([c, m.expand_as(c), raw, out, y], 1).flatten(0, 1)
            save_image(panels[:30], str(save_to), nrow=5)
            save_to = None

    return {s: {k: (v if k == 'n' else v / d['n']) for k, v in d.items()} for s, d in sums.items()}


def atomic_save(obj, target):
    tmp = target.with_name(target.name + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, target)


start_epoch, best, history = 0, float('inf'), []
if RESUME and (RUN / 'last.pt').exists():
    # Only load checkpoints you created/trust (contains Python RNG objects).
    ck = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['cfg'] == CFG and ck['fingerprint'] == fingerprint, 'Config/data changed; use new RUN'
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)
    scheduler.load_state_dict(ck['scheduler'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch, best, history = ck['epoch'], ck['best'], ck['history']
    torch.set_rng_state(ck['torch_rng'])
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
    print('Resume completed epoch', start_epoch)

# Smoke test does not update weights or optimizer.
c, m, y, _ = next(iter(loader('val')))
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    smoke = model(torch.cat([c.to(DEVICE), m.to(DEVICE)], 1))
assert smoke.shape == y.shape and torch.isfinite(smoke).all()
del smoke

environment = dict(
    cfg=CFG,
    fingerprint=fingerprint,
    official_counts=official_counts,
    gpu=torch.cuda.get_device_name(),
    torch=torch.__version__,
    python=platform.python_version(),
)
(RUN / 'config.json').write_text(json.dumps(environment, indent=2))
(RUN / 'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze']).decode())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ImageHarmonization/runs/unet_masked_l1_all4_bs32_seed42/config.json'

Chuyển RUN sang unet_full_l1_all4_bs32_seed42, nâng batch_size=32, lr=3e-4 và khởi tạo torch.amp.GradScaler('cuda') để huấn luyện Mixed Precision (float16).

In [ ]:
RUN = PROJECT / 'runs/unet_full_l1_all4_bs32_seed42'
CFG = dict(
    seed=42,
    epochs=50,
    size=256,
    batch_size=32,       # Tốn khoảng ~6.5 GB / 15 GB VRAM T4
    workers=2,           # Giữ nguyên 2 cho Colab
    base_channels=32,
    lr=3e-4,             # Tăng nhẹ từ 2e-4 -> 3e-4 cho khớp với batch 32
    val_fraction=0.05,
    loss='raw_full_image_l1',
    architecture='unet_groupnorm_v1',
)

In [ ]:
scaler = torch.amp.GradScaler('cuda')

Huấn luyện mô hình từ Epoch 1 đến Epoch 50 bằng hàm mất mát nn.functional.l1_loss(raw, y) trên toàn bộ pixel ảnh, cắt gradient tối đa 1.0, đánh giá trên tập val sau mỗi epoch và lưu best.pt, last.pt, history.json. Kết quả tốt nhất trên tập validation đạt fMSE = 556.07 (tại Epoch 39).

In [ ]:
RUN.mkdir(parents=True, exist_ok=True)
for epoch in range(start_epoch, CFG['epochs']):
    torch.cuda.synchronize()
    tic = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    model.train()
    total, count = 0., 0

    bar = tqdm(loader('train', epoch), desc=f'Epoch {epoch+1}/{CFG["epochs"]}')
    for c, m, y, _ in bar:
        c, m, y = [x.to(DEVICE, non_blocking=True) for x in (c, m, y)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            raw = model(torch.cat([c, m], 1))
            loss = nn.functional.l1_loss(raw, y)  # ALL RGB pixels, NO mask blending!
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite loss; inspect data/AMP')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item() * c.shape[0]
        count += c.shape[0]
        bar.set_postfix(L1=f'{total/count:.5f}')

    # Lưu ý: Hàm loader() ở cell trước phải để drop_last=False để dòng này không lỗi
    assert count == len(records['train']), f'Not all training samples were seen ({count} vs {len(records["train"])})'

    scores = evaluate(loader('val'), save_to=RUN / f'val_{epoch+1:03d}.png')
    improved = scores['all']['fMSE'] < best
    best = min(best, scores['all']['fMSE'])
    lr_used = optimizer.param_groups[0]['lr']
    scheduler.step()
    torch.cuda.synchronize()

    row = dict(
        epoch=epoch + 1,
        train_L1=total / count,
        val=scores,
        lr=lr_used,
        seconds=time.perf_counter() - tic,
        peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
    )
    history.append(row)

    ck = dict(
        epoch=epoch + 1,
        model=model.state_dict(),
        optimizer=optimizer.state_dict(),
        scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(),
        cfg=CFG,
        fingerprint=fingerprint,
        best=best,
        history=history,
        torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all(),
    )

    if improved:
        atomic_save(
            dict(model=model.state_dict(), cfg=CFG, epoch=epoch + 1, fingerprint=fingerprint, val=scores),
            RUN / 'best.pt',
        )
    atomic_save(ck, RUN / 'last.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
    print(json.dumps(row, ensure_ascii=False))

print('Done. Best validation fMSE (0..255 scale):', best)

Epoch 2/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"epoch": 2, "train_L1": 0.02071980167703163, "val": {"all": {"n": 3279, "raw_L1": 0.01955097592607613, "MSE": 87.1628373135296, "fMSE": 792.3299146270053, "PSNR": 33.380636994378456, "fPSNR": 21.25767867104896}, "HCOCO": {"n": 1924, "raw_L1": 0.017154567382441692, "MSE": 52.64224946895409, "fMSE": 777.8600748304023, "PSNR": 34.629396507001466, "fPSNR": 20.94372661297138}, "HAdobe5k": {"n": 974, "raw_L1": 0.019011304945182016, "MSE": 82.24178635241805, "fMSE": 601.5462164281574, "PSNR": 33.18059463128907, "fPSNR": 22.92539432357224}, "HFlickr": {"n": 368, "raw_L1": 0.03307426296700926, "MSE": 276.1132125848013, "fMSE": 1356.597201347351, "PSNR": 27.559945619624592, "fPSNR": 18.587819447983865}, "Hday2night": {"n": 13, "raw_L1": 0.0318402022553178, "MSE": 216.16103341029242, "fMSE": 1254.8631685697114, "PSNR": 28.322358938363884, "fPSNR": 18.349748538090633}}, "lr": 0.0002997049959000266, "seconds": 880.567958829999, "peak_VRAM_GB": 11.63838815689087}


Epoch 3/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"epoch": 3, "train_L1": 0.01907387090991195, "val": {"all": {"n": 3279, "raw_L1": 0.017299149746087202, "MSE": 89.48838951243027, "fMSE": 821.1637843624417, "PSNR": 33.37246138091929, "fPSNR": 21.2742922699426}, "HCOCO": {"n": 1924, "raw_L1": 0.01455866415857138, "MSE": 54.17582437011903, "fMSE": 810.6604556977873, "PSNR": 34.534990647726396, "fPSNR": 20.854623743997045}, "HAdobe5k": {"n": 974, "raw_L1": 0.017142179870530502, "MSE": 83.05258275057501, "fMSE": 615.9248505118446, "PSNR": 33.37879443413423, "fPSNR": 23.19505965195642}, "HFlickr": {"n": 368, "raw_L1": 0.031478143454043435, "MSE": 284.60576409943724, "fMSE": 1396.6651414580967, "PSNR": 27.488121903460957, "fPSNR": 18.521057468393575}, "Hday2night": {"n": 13, "raw_L1": 0.03327709164183874, "MSE": 274.6158719613002, "fMSE": 1461.6735159067007, "PSNR": 27.41570942218487, "fPSNR": 17.413155409005974}}, "lr": 0.00029882114784651444, "seconds": 880.9755324389989, "peak_VRAM_GB": 11.63838815689087}


... (152 outputs left out)

{"epoch": 49, "train_L1": 0.009885982774250419, "val": {"all": {"n": 3279, "raw_L1": 0.011285657473331696, "MSE": 59.726156824988976, "fMSE": 558.075809288403, "PSNR": 34.747577298156514, "fPSNR": 22.85466265787472}, "HCOCO": {"n": 1924, "raw_L1": 0.00876418859049244, "MSE": 35.113584032363555, "fMSE": 539.3781362858731, "PSNR": 35.92352019327842, "fPSNR": 22.473369719332815}, "HAdobe5k": {"n": 974, "raw_L1": 0.012165331247298992, "MSE": 62.07684717928604, "fMSE": 433.2943249742598, "PSNR": 34.67560848807897, "fPSNR": 24.708916635728716}, "HFlickr": {"n": 368, "raw_L1": 0.02189364590183528, "MSE": 180.15744427753532, "fMSE": 965.7968640534774, "PSNR": 28.97457111140956, "fPSNR": 20.058332093383957}, "Hday2night": {"n": 13, "raw_L1": 0.018267513861736424, "MSE": 117.13414647028996, "fMSE": 1132.7019958496094, "PSNR": 29.520636338454025, "fPSNR": 19.517269354600174}}, "lr": 2.1788521534855743e-06, "seconds": 903.6431501970001, "peak_VRAM_GB": 6.01436185836792}


Epoch 50/50:   0%|          | 0/1952 [00:00<?, ?it/s]

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

{"epoch": 50, "train_L1": 0.009876236876400159, "val": {"all": {"n": 3279, "raw_L1": 0.011283519639048218, "MSE": 59.80749830135524, "fMSE": 558.3095658998294, "PSNR": 34.74801247990251, "fPSNR": 22.85595115185802}, "HCOCO": {"n": 1924, "raw_L1": 0.008768671908751788, "MSE": 35.22497310156386, "fMSE": 540.3394418783843, "PSNR": 35.9199840958054, "fPSNR": 22.470196998540676}, "HAdobe5k": {"n": 974, "raw_L1": 0.012152118672658852, "MSE": 62.1114101382374, "fMSE": 433.5771340481799, "PSNR": 34.681201862358705, "fPSNR": 24.715845594660703}, "HFlickr": {"n": 368, "raw_L1": 0.02189048254892797, "MSE": 180.2549370151499, "fMSE": 962.712667910949, "PSNR": 28.97952460206073, "fPSNR": 20.065177031185318}, "Hday2night": {"n": 13, "raw_L1": 0.018144229140419226, "MSE": 115.81602973204393, "fMSE": 1115.5069274902344, "PSNR": 29.594450290386494, "fPSNR": 19.598926544189453}}, "lr": 1.295004099973402e-06, "seconds": 886.9632898490017, "peak_VRAM_GB": 6.01436185836792}
Done. Best validation fMSE (0..2

Các cell phụ được dùng tạm ở lần chạy đầu tiên để tạo thư mục trên Drive và lưu thủ công kết quả của Epoch 1 đang nằm trong bộ nhớ GPU trước khi chạy nối tiếp từ Epoch 2 đến 50.

In [ ]:
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
RUN.mkdir(parents=True, exist_ok=True)
print("Đã tạo thư mục:", RUN)

Đã tạo thư mục: /content/drive/MyDrive/ImageHarmonization/runs/unet_full_l1_all4_bs32_seed42


In [ ]:
# Đánh giá và lưu lại kết quả của Epoch 1 đang nằm sẵn trong GPU
RUN.mkdir(parents=True, exist_ok=True)
scores = evaluate(loader('val'), save_to=RUN / 'val_001.png')
improved = scores['all']['fMSE'] < best
best = min(best, scores['all']['fMSE'])
lr_used = optimizer.param_groups[0]['lr']
scheduler.step()

row = dict(
    epoch=1,
    train_L1=total / count,
    val=scores,
    lr=lr_used,
    seconds=0.0,
    peak_VRAM_GB=torch.cuda.max_memory_allocated() / 2**30,
)
history.append(row)

ck = dict(
    epoch=1,
    model=model.state_dict(),
    optimizer=optimizer.state_dict(),
    scheduler=scheduler.state_dict(),
    scaler=scaler.state_dict(),
    cfg=CFG,
    fingerprint=fingerprint,
    best=best,
    history=history,
    torch_rng=torch.get_rng_state(),
    cuda_rng=torch.cuda.get_rng_state_all(),
)
if improved:
    atomic_save(dict(model=model.state_dict(), cfg=CFG, epoch=1, fingerprint=fingerprint, val=scores), RUN / 'best.pt')
atomic_save(ck, RUN / 'last.pt')
(RUN / 'history.json').write_text(json.dumps(history, indent=2))
start_epoch = 1
print("Đã lưu xong Epoch 1! Val fMSE:", scores['all']['fMSE'])

evaluate:   0%|          | 0/103 [00:00<?, ?it/s]

Đã lưu xong Epoch 1! Val fMSE: 837.0025404848387


Bật RUN_TEST = True, nạp best.pt của nhánh Full L1 và chấm điểm trên toàn bộ tập Test, lưu kết quả vào test_metrics.json và ảnh mẫu test_preview.png. Kết quả đạt được trên toàn tập Test (all): MSE = 60.59, fMSE = 586.10, PSNR = 34.61 dB, fPSNR = 22.55 dB, SSIM = 0.9815.

In [ ]:
RUN_TEST=True

In [ ]:
if RUN_TEST:
    ck = torch.load(RUN/'best.pt',map_location=DEVICE,weights_only=False)
    model.load_state_dict(ck['model'])
    test_scores = evaluate(loader('test'),save_to=RUN/'test_preview.png',include_ssim=True)
    (RUN/'test_metrics.json').write_text(json.dumps(test_scores,indent=2))
    print(json.dumps(test_scores,indent=2))
else:
    print('Test skipped. Set RUN_TEST=True only after finalizing the experiment.')



evaluate:   0%|          | 0/232 [00:00<?, ?it/s]

{
  "all": {
    "n": 7404,
    "raw_L1": 0.011445396991563847,
    "MSE": 60.589480016946084,
    "fMSE": 586.0978380679311,
    "PSNR": 34.60869808083673,
    "fPSNR": 22.552497441030592,
    "SSIM": 0.9815317429783279
  },
  "HCOCO": {
    "n": 4283,
    "raw_L1": 0.009607572722249385,
    "MSE": 41.699399891591035,
    "fMSE": 616.4832304233762,
    "PSNR": 35.25228586827041,
    "fPSNR": 21.88640418198519,
    "SSIM": 0.9864938430855159
  },
  "HAdobe5k": {
    "n": 2160,
    "raw_L1": 0.012143268062694309,
    "MSE": 63.043995355235204,
    "fMSE": 367.59394594210164,
    "PSNR": 35.02502899434831,
    "fPSNR": 24.94526985353894,
    "SSIM": 0.9770823446136934
  },
  "HFlickr": {
    "n": 828,
    "raw_L1": 0.019324031481879275,
    "MSE": 150.45984267079888,
    "fMSE": 953.6538896652811,
    "PSNR": 30.025124654677754,
    "fPSNR": 19.981911400090095,
    "SSIM": 0.9687137542571422
  },
  "Hday2night": {
    "n": 133,
    "raw_L1": 0.010246077246081672,
    "MSE": 69.5503805342